# Portfolio Project: Online Retail Exploratory Data Analysis with Python

## Overview

In this project, you will step into the shoes of an entry-level data analyst at an online retail company, helping interpret real-world data to help make a key business decision.

## Case Study
In this project, you will be working with transactional data from an online retail store. The dataset contains information about customer purchases, including product details, quantities, prices, and timestamps. Your task is to explore and analyze this dataset to gain insights into the store's sales trends, customer behavior, and popular products. 

By conducting exploratory data analysis, you will identify patterns, outliers, and correlations in the data, allowing you to make data-driven decisions and recommendations to optimize the store's operations and improve customer satisfaction. Through visualizations and statistical analysis, you will uncover key trends, such as the busiest sales months, best-selling products, and the store's most valuable customers. Ultimately, this project aims to provide actionable insights that can drive strategic business decisions and enhance the store's overall performance in the competitive online retail market.

## Project Objectives
1. Describe data to answer key questions to uncover insights
2. Gain valuable insights that will help improve online retail performance
3. Provide analytic insights and data-driven recommendations

## Dataset

The dataset you will be working with is the "Online Retail" dataset. It contains transactional data of an online retail store from 2010 to 2011. The dataset is available as a .xlsx file named `Online Retail.xlsx`. This data file is already included in the Coursera Jupyter Notebook environment, however if you are working off-platform it can also be downloaded [here](https://archive.ics.uci.edu/ml/machine-learning-databases/00352/Online%20Retail.xlsx).

The dataset contains the following columns:

- InvoiceNo: Invoice number of the transaction
- StockCode: Unique code of the product
- Description: Description of the product
- Quantity: Quantity of the product in the transaction
- InvoiceDate: Date and time of the transaction
- UnitPrice: Unit price of the product
- CustomerID: Unique identifier of the customer
- Country: Country where the transaction occurred

## Tasks

You may explore this dataset in any way you would like - however if you'd like some help getting started, here are a few ideas:

1. Load the dataset into a Pandas DataFrame and display the first few rows to get an overview of the data.
2. Perform data cleaning by handling missing values, if any, and removing any redundant or unnecessary columns.

## Task 1: Load the Data

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path = Path.cwd()) -> Path:
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "data").exists():
            return path
    raise FileNotFoundError("Project root not found")

BASE_DIR = find_project_root()
print(f'Working base set at {BASE_DIR}')

if str(BASE_DIR) not in sys.path:
    sys.path.append(str(BASE_DIR))

DATA_DIR = BASE_DIR / "data"
retail_path = DATA_DIR / 'Online Retail.xlsx'
output_path = DATA_DIR / 'Online Retail Cleaned.xlsx'

In [ ]:
import pandas as pd
import numpy as np

import src.utils.data_exploring as expl
import src.utils.data_imputation as imp
import src.utils.data_cleaning as cln

In [ ]:
retail_path

In [ ]:
df = pd.read_excel(retail_path)
display(df.head(5))
display(df.tail(5))

In [ ]:
expl.show_basic_df_info(df)

In [ ]:
print('\nId counts:')
print(f'different invoices: {df.InvoiceNo.nunique()}')
print(f'different customers: {df.CustomerID.nunique()}')
print(f'different products: {df.StockCode.nunique()}')
print(f'different descriptions: {df.Description.nunique()}')
print(f'duplicated rows {df.duplicated().sum()}')

In [ ]:
expl.get_categorical_counts(df=df, cat_cols=['Country', 'StockCode'])

In [ ]:
# assuming product returns
display(df[df['Quantity'] <= 0])

In [ ]:
display(df[df['Quantity'] == 0])

In [ ]:
# assuming these are broken
display(df[df['UnitPrice'] <= 0])

In [ ]:
display(df[df['UnitPrice'] < 0])

## Task 2: Data cleaning
<b>Step 1:</b> Drop duplicated rows

In [ ]:
df_clean = df.drop_duplicates().copy()

<b>Step 2:</b> Standardize descriptions

In [ ]:
df_clean['CustomerID'] = (
    df_clean['CustomerID']
    .astype("Int64")
    .astype("string")
    .fillna("missing")
)
df_clean = cln.standardize_strs(df=df_clean, str_cols=['Description', 'InvoiceNo', 'StockCode', 'Country'])

### Data imputation
<b>Step 1:</b> Attempting to complete descriptions with product ids

In [ ]:
df_imp = imp.fill_and_standardize_values_by_id(
    df_clean, id_col='StockCode',
    value_col='Description', standardize_existing=True
)
print(f'{df["Description"].isna().sum()} vs {df_imp["Description"].isna().sum()}')

<b>Step 2:</b> Attempting to complete unit price with product ids

In [ ]:
df_imp['UnitPrice'] = df_imp['UnitPrice'].replace(0.0, np.nan)
df_imp['UnitPrice'].isna().sum()

In [ ]:
df_imp_2 = imp.fill_and_standardize_values_by_id(df_imp, id_col='StockCode', value_col='UnitPrice')
print(f'{df_imp["UnitPrice"].isna().sum()} vs {df_imp_2["UnitPrice"].isna().sum()}')

<b>Step 3:</b> Removing rows without unit price

Too few rows to matter, and UnitPrice is one of the most interesting variables

In [ ]:
df_imp_2 = df_imp_2.dropna(subset=['UnitPrice'])

In [ ]:
len(df_imp_2['UnitPrice'])

<b>Step 4:</b> Replacing missing descriptions with missing

Too few rows to matter, and description is one of the least interesting variables

In [ ]:
df_imp_2['Description'] = df_imp_2['Description'].fillna(value='missing')

<b>Step 5:</b> Replacing missing customer ids with missing

There is no reasonable way to impute, and the amount of rows is too considerable to drop

In [ ]:
df_imp_2['CustomerID'] = df_imp_2['CustomerID'].fillna(value='missing')

### Saving clean dataset

In [ ]:
df_imp_2.to_excel(output_path, index=False)

# Cleaning conclusions

- The raw dataset contains duplicated rows, missing product descriptions, many missing customer IDs, zero or negative unit prices, and negative quantities. These issues need to be handled before business analysis because they can distort product popularity, customer-level analysis, and revenue calculations.

- Duplicated rows were removed (5268 rows) to avoid double-counting transactions. Product descriptions were standardized by StockCode using the most frequent description, assuming that each StockCode represents one product. 

- After replacing zero UnitPrice values with missing values, 2,510 rows required price imputation. Most were completed using the most frequent valid UnitPrice for the same StockCode; 132 rows still had no valid reference price and were removed.

- . These records were kept for now because they likely represent returns or cancellations rather than simple data errors.

- CustomerID has a large amount of missing data (135080 rows). These values were not imputed because there is no reliable way to infer customer identity from the available columns. They were kept as "missing" so transaction-level analysis remains possible, while customer-level analysis can explicitly exclude or compare unidentified customers.

- There are 10,624 rows with negative quantities and no rows with Quantity equal to zero. Negative quantities and negative prices were not treated as normal outliers. They likely represent returns, cancellations, discounts, or accounting adjustments, and should be analyzed separately depending on the business question.